# 01. ⑤ 인코더 학습 (④ 합성 데이터)

합성 데이터는 학습 중 GPU에서 바로 만든다. 체크포인트는 Drive(`runs/encoder/`)에 저장되어 세션이 끊겨도 이어서 학습한다.
목표: Colab GPU에서 몇 시간 이내. 넘으면 `EncoderConfig.channels`나 `n_mels`를 줄인다.

In [ ]:
# 공통 준비: Drive 연결, 저장소 받기, 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE = '/content/drive/MyDrive/song2nam'   # 데이터·실행 기록·체크포인트를 둘 Drive 폴더
REPO = '/content/song2nam'
BRANCH = 'main'                               # 작업 브랜치로 바꿔서 쓸 수 있음
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
# data/, runs/를 Drive에 연결해서 세션이 끊겨도 남게 한다
for d in ['data', 'runs']:
    if os.path.islink(d):
        continue
    subprocess.run(['rm', '-rf', d])
    os.symlink(f'{DRIVE}/{d}', d)

In [ ]:
# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
!grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import torch
from ampcap.synth_data import di_reader_for_split, SynthGenerator, SynthConfig
from ampcap.encoder import Encoder, EncoderConfig, TrainConfig, train

EGDB = 'data/EGDB/audio_DI'
SPLIT = 'data/egdb_split.json'
dev = 'cuda'
train_gen = SynthGenerator(di_reader_for_split(EGDB, 'train', SPLIT), dev, SynthConfig(batch_size=16), seed=0)
val_gen = SynthGenerator(di_reader_for_split(EGDB, 'val', SPLIT), dev, SynthConfig(batch_size=16), seed=1)

# 생성 속도 확인 (④ 실시간 생성 가능 여부)
import time; t = time.time(); b = train_gen.next_batch(); torch.cuda.synchronize(); print('batch', time.time() - t, 's')

In [ ]:
model = Encoder(EncoderConfig())
cfg = TrainConfig(steps=20000, max_hours=3.5)
hist = train(model, train_gen, val_gen, cfg, ckpt_path='runs/encoder/encoder.pt', device=dev)
hist['best_val']

In [ ]:
import matplotlib.pyplot as plt
h = [r for r in hist['history'] if 'theta_mse' in r]
v = [r for r in hist['history'] if 'val' in r]
plt.semilogy([r['step'] for r in h], [r['theta_mse'] for r in h], label='train θ MSE')
plt.semilogy([r['step'] for r in v], [r['val'] for r in v], label='val θ MSE')
plt.legend(); plt.savefig('runs/encoder/curve.png')

체크포인트: `runs/encoder/encoder_best.pt` → 02, 03 노트북과 앱의 `--encoder`로 쓴다.